# Agentic MDS — one-button live run

**Do not paste this `.ipynb` file into a code cell.** It is JSON. That causes `NameError: name 'true' is not defined`.

Open it as a notebook:
- [Open Colab_Start_Here.ipynb in Google Colab](https://colab.research.google.com/github/rockyforever8-sys/Agentic-MDS/blob/main/Colab_Start_Here.ipynb)
- Or Colab **File → Upload notebook**

This notebook runs the **original IMDS agent** (`imds_agent_v2.py`) — same XPaths and actions that already produced your Excel output. The only change is **secret authentication**: passwords stay in Colab 🔑, not in the script.

| Secret | Purpose |
|---|---|
| `IMDS_USERNAME` | IMDS login |
| `IMDS_PASSWORD` | IMDS password |
| `OTP_SECRET` | Authenticator TOTP seed (not a Gmail app password) |
| `IMDS_MASTER_KEY` | Optional passphrase for the encrypted Drive vault |

Optional: `NUM_ITERATIONS` (default **3**), `RECIPIENT_COMPANY_IDS` (default `9994,293798`).

Then click **Run IMDS until complete**. Output: `imds_output/check_summary.xlsx`.


In [ ]:
# Cell 1 — clone the original agent and install Chromium OS libraries.
import os, pathlib, subprocess, sys

ROOT = pathlib.Path("/content/Agentic-MDS")
REPO = "https://github.com/rockyforever8-sys/Agentic-MDS.git"
REF = os.environ.get("IMDS_GIT_REF", "main")
if not (ROOT / ".git").exists():
    try:
        subprocess.check_call(["git", "clone", "--depth", "1", "--branch", REF, REPO, str(ROOT)])
    except subprocess.CalledProcessError:
        subprocess.check_call(["git", "clone", "--depth", "1", REPO, str(ROOT)])
else:
    fetched = False
    for _ref in (REF, "main"):
        try:
            subprocess.check_call(["git", "-C", str(ROOT), "fetch", "--depth", "1", "origin", _ref])
            subprocess.check_call(["git", "-C", str(ROOT), "checkout", "-B", _ref, f"origin/{_ref}"])
            fetched = True
            break
        except subprocess.CalledProcessError:
            print("Could not fetch origin/" + _ref)
    if not fetched:
        raise RuntimeError("git fetch failed")
os.chdir(ROOT)
print("Working directory:", os.getcwd())
print("git HEAD:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

%pip install -q playwright pandas openpyxl nest_asyncio pyotp cryptography ipywidgets
!python -m playwright install-deps chromium
!python -m playwright install chromium
from pathlib import Path as _P
print("libatk present:", _P("/usr/lib/x86_64-linux-gnu/libatk-1.0.so.0").exists())
print("Install done.")


In [ ]:
# Cell 2 — compile only. No IMDS login.
!python -m py_compile imds_decisions.py imds_secrets.py imds_agent_v2.py
print("compile OK")


In [ ]:
# Cell 3 — one button. Set Colab Secrets first (key icon, left sidebar).
# Playwright Sync API cannot start in Colab's asyncio loop; the button runs a subprocess.
import os
from pathlib import Path
from IPython.display import display
import ipywidgets as widgets

try:
    from google.colab import userdata, drive
    for _key in (
        "IMDS_USERNAME", "IMDS_PASSWORD", "OTP_SECRET", "IMDS_MASTER_KEY",
        "IMDS_CONTACT_NAME", "RECIPIENT_COMPANY_IDS", "NUM_ITERATIONS",
    ):
        try:
            val = userdata.get(_key)
            if val:
                os.environ[_key] = val
        except Exception:
            pass
    if not Path("/content/drive/MyDrive").exists():
        try:
            drive.mount("/content/drive")
        except Exception:
            pass
except ImportError:
    pass

os.environ.setdefault("NUM_ITERATIONS", "3")
os.environ.setdefault("RECIPIENT_COMPANY_IDS", "9994,293798")

from imds_secrets import apply_stored_credentials, missing_secret_keys
apply_stored_credentials(persist=True)

run_btn = widgets.Button(
    description="Run IMDS until complete",
    button_style="success",
    layout=widgets.Layout(width="280px", height="48px"),
)
out = widgets.Output()


def _on_run(_):
    with out:
        out.clear_output()
        apply_stored_credentials(persist=True)
        missing = missing_secret_keys()
        if missing:
            raise RuntimeError(
                "Private secrets missing: " + ", ".join(missing) +
                ". Add IMDS_USERNAME, IMDS_PASSWORD, OTP_SECRET in Colab Secrets."
            )
        import subprocess, sys
        env = os.environ.copy()
        env["PYTHONUNBUFFERED"] = "1"
        proc = subprocess.Popen(
            [sys.executable, "-u", "imds_agent_v2.py"],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            env=env,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
        rc = proc.wait()
        print("exit code", rc)
        report = Path("imds_output/check_summary.xlsx")
        if report.exists():
            try:
                import pandas as pd
                from IPython.display import display as show
                show(pd.read_excel(report))
            except Exception:
                print("Wrote", report)


run_btn.on_click(_on_run)
display(run_btn, out)
print("Secrets loaded:", not bool(missing_secret_keys()), "| click the green button")
